# Add Style — dev cockpit

Thin UI over `training/add_style_helper.py`.  
Run the cells top-to-bottom. Each cell has one job; re-run any cell independently if you change a value.

In [1]:
# Cell 1 – bootstrap: locate repo root and load catalog
import sys, pathlib

# Make sure the training/ helper is importable
_here = pathlib.Path().resolve()
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

from add_style_helper import setup

ctx = setup()


Repository     : C:\Users\i09300076\OneDrive - Endress+Hauser\DEV\Python3\style_transfer
Existing styles: ['abstract', 'akira_manga', 'anime', 'candy', 'cubism', 'escher', 'hundertwasser', 'hundertwasserworks', 'mosaic', 'mucha', 'muchaworks', 'rain_princess', 'starry_night', 'style_cezanne', 'style_monet', 'style_ukiyoe', 'style_vangogh', 'tintin']


In [2]:
# Cell 2 – pick the .onnx file via file dialog
from add_style_helper import pick_onnx_file

paths = pick_onnx_file()


.onnx : C:\Users\i09300076\Downloads\model.onnx
.pth  : C:\Users\i09300076\Downloads\model.pth


In [3]:
# Cell 3 – validate model files
from add_style_helper import report_model_files

report_model_files(paths.onnx_path, paths.pth_path, paths.data_path)


  .onnx      OK  (6.8 MB)
  .pth       OK  (6.7 MB)
  .onnx.data -- not present (weights embedded in .onnx)


In [4]:
# Cell 4 – fill in style metadata
# ── EDIT THESE VALUES ──────────────────────────────────────────────────────
STYLE_NAME   = "Kusama"          # display name shown in the gallery
STYLE_DESC   = "Yayoi Kusama"
STYLE_AUTHOR = "PeterWazinski"
TENSOR_LAYOUT = "nchw"             # "nchw"  or  "nhwc_tanh"
# ───────────────────────────────────────────────────────────────────────────

from add_style_helper import validate_style_id

style_id, msg = validate_style_id(STYLE_NAME, ctx.existing_ids)
print(msg)


OK  ID will be: 'kusama'


In [5]:
# Cell 5 – install: copy files, generate preview, update catalog.json
from add_style_helper import install_style

new_id = install_style(
    onnx_path=paths.onnx_path,
    pth_path=paths.pth_path,
    data_path=paths.data_path,
    style_name=STYLE_NAME,
    style_desc=STYLE_DESC,
    style_author=STYLE_AUTHOR,
    tensor_layout=TENSOR_LAYOUT,
    styles_dir=ctx.styles_dir,
    catalog_path=ctx.catalog_path,
    catalog=ctx.catalog,
    existing_ids=ctx.existing_ids,
    content_image=ctx.content_image,
    repo_root=ctx.repo_root,
)


Copied model.pth
Preview generated: C:\Users\i09300076\OneDrive - Endress+Hauser\DEV\Python3\style_transfer\styles\kusama\preview.jpg

OK  'Kusama'  (id='kusama', layout='nchw')  added to gallery.
Files:
  model.onnx  (6.8 MB)
  model.pth  (6.7 MB)
  preview.jpg  (0.0 MB)

Next: git add -A ; git commit -m 'feat: add <name> style'


In [ ]:
# Cell 6 – copy new style + catalog.json into dist\
import shutil

dist_styles = ctx.repo_root / "dist" / "PetersPictureStyler" / "styles"
src_style_dir = ctx.styles_dir / new_id

if dist_styles.exists():
    dst_style_dir = dist_styles / new_id
    shutil.copytree(src_style_dir, dst_style_dir, dirs_exist_ok=True)
    shutil.copy2(ctx.catalog_path, dist_styles / "catalog.json")
    print(f"Copied {new_id}/ and catalog.json to {dist_styles}")
else:
    print(f"dist styles dir not found ({dist_styles}) -- run compile.ps1 first.")


In [6]:
# Cell 6 – commit
import subprocess

msg = f"feat: add {new_id} style"
result = subprocess.run(
    ["git", "add", "-A"],
    cwd=str(ctx.repo_root), capture_output=True, text=True
)
result2 = subprocess.run(
    ["git", "commit", "-m", msg],
    cwd=str(ctx.repo_root), capture_output=True, text=True
)
print(result2.stdout or result2.stderr)


[main 0b26c58] feat: add kusama style
 Committer: Peter Wazinski <peter.wazinski@endress.com>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly:

    git config --global user.name "Your Name"
    git config --global user.email you@example.com

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 7 files changed, 154 insertions(+)
 create mode 100644 styles/kusama/model.onnx
 create mode 100644 styles/kusama/model.pth
 create mode 100644 styles/kusama/preview.jpg
 create mode 100644 training/add_style_helper.ipynb

